# Region classifier (spine / hip_left / hip_right) — обучение

Ноутбук для Kaggle/Colab (локально GPU нет). Обучает лёгкий CNN-классификатор
(3 класса: `spine`, `hip_left`, `hip_right`) на DXA-снимках поясничного отдела
позвоночника и проксимального отдела бедра — Шаг 1 плана (`docs/plan.md`).

## Перед запуском

Загрузите единый пакет, собранный `build_training_package.py`
(из исходной `Исследования` + `manifest.csv`), — плоская папка:

```
training_package/
  images/*.png   (252 PNG, имя файла = image_hash)
  manifest.csv   (колонка image_filename вместо вложенных путей)
```

Загрузите эту папку целиком как один Kaggle Dataset (или положите в
Google Drive для Colab) и пропишите `IMAGES_DIR`/`MANIFEST_PATH` в ячейке
конфигурации ниже. DICOM и pydicom ноутбуку больше не нужны — вся
конвертация уже сделана на этапе сборки пакета.

## Важно про метки

Метки `region`/`side` в `manifest.csv` — это не разметка организаторов
(per-image разметки зоны в датасете нет), а результат эвристического
OpenCV-классификатора (`services/region_classifier/classify.py`),
согласованный на 99.6% с независимой эвристикой по ширине снимка и
дополнительно проверенный на маленькой выборке с именами файлов
(3/3 совпадений) и ручным просмотром по классам (`export_for_review.py`,
найденные ошибки region/side исправлены в манифесте вручную).

## Воспроизводимость

Ноутбук фиксирует: seed для всех источников случайности (Python/NumPy/
PyTorch/CUDA/albumentations), детерминированные алгоритмы cuDNN, версии
всех ключевых библиотек (сохраняются в `run_info.json` вместе с
чекпоинтами), и хэш `manifest.csv`, на котором обучалась модель — чтобы
можно было убедиться, что при повторном запуске используются те же
данные и то же окружение. Полный bit-exact детерминизм на GPU не
гарантирован (некоторые CUDA-операции недетерминированы даже при
выставленных флагах — известное ограничение PyTorch, не баг этого
ноутбука), но воспроизводимость метрик с точностью до статистического
шума обеспечена.


---

## Версия с голосованием критериев

После обычной оценки на тесте: регион решают **три критерия голосованием** — сеть (одна модель), математика (маска кости: CLAHE + Otsu, число периодических пиков ширины кости, как в `classify.py`) и ширина снимка (300 — позвоночник, 280 — бедро). Показаны точность каждого критерия отдельно и итог голосования.


In [19]:
# Установка зависимостей. Версии пин-нутые для воспроизводимости —
# если Kaggle/Colab ругается на конфликт версий, ослабьте пины, но тогда
# обязательно зафиксируйте фактически установленные версии в run_info.json
# ниже (эта ячейка это делает автоматически после установки).
# pydicom не нужен — training_package уже содержит готовые PNG.
!pip install -q albumentations timm scikit-learn


In [20]:
import os
import io
import json
import random
import hashlib
import platform
import subprocess
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import cv2
import albumentations as A
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import timm
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix


## Воспроизводимость: seed + детерминизм

In [21]:
SEED = 42


def set_full_determinism(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # cuDNN: детерминированные (но чуть более медленные) алгоритмы вместо
    # автотюнинга, который сам по себе недетерминирован между запусками.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_full_determinism(SEED)


def worker_init_fn(worker_id: int) -> None:
    # без этого воркеры DataLoader'а получают собственное недетерминированное
    # состояние RNG (albumentations внутри Dataset.__getitem__ использует
    # глобальный random/np.random, который в отдельном процессе-воркере
    # инициализируется по-своему без явного seed)
    worker_seed = SEED + worker_id
    random.seed(worker_seed)
    np.random.seed(worker_seed)


def make_generator(seed: int = SEED) -> torch.Generator:
    g = torch.Generator()
    g.manual_seed(seed)
    return g


## Конфигурация путей и гиперпараметров

In [22]:
# --- ПРАВЬТЕ ПОД СВОЮ СРЕДУ ---
# Kaggle пример (после загрузки training_package как Kaggle Dataset):
#   IMAGES_DIR = "/kaggle/input/dxa-training-package/images"
#   MANIFEST_PATH = "/kaggle/input/dxa-training-package/manifest.csv"
# Colab пример (после подключения Google Drive):
#   IMAGES_DIR = "/content/drive/MyDrive/dxa/training_package/images"
#   MANIFEST_PATH = "/content/drive/MyDrive/dxa/training_package/manifest.csv"

IMAGES_DIR = "/kaggle/input/datasets/cgvhbjmk/dxa-data/training_package/images"
MANIFEST_PATH = "/kaggle/input/datasets/cgvhbjmk/dxa-data/training_package/manifest.csv"
OUTPUT_DIR = "./outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

IMAGE_SIZE = 224
BATCH_SIZE = 16
NUM_EPOCHS = 40
EARLY_STOPPING_PATIENCE = 7
HEAD_LR = 1e-3
BACKBONE_LR = 1e-5
N_FOLDS = 5  # должно совпадать с тем, что использовалось в build_manifest.py
TEST_FOLD = N_FOLDS - 1  # этот fold НИКОГДА не используется в обучении/CV —
# это настоящий отложенный тест, а не просто ещё один fold для валидации.
# Метки во всём manifest.csv (включая этот fold) вручную перепроверены по
# экспорту export_for_review.py, поэтому это честная human-verified оценка,
# а не просто сверка с прокси-эвристикой.
NUM_WORKERS = 2
FINAL_EPOCHS = 25  # ОДНА модель на всех данных (фолды 0-3), фиксированное число эпох без ранней остановки (как у классификатора подвздошных); эпох выбрано по прежним CV-прогонам: плато к 20-28

# Аблация: отключить агрессивную геометрическую/шумовую аугментацию,
# чтобы проверить гипотезу, что именно она (в первую очередь CoarseDropout,
# закрывающий крыло подвздошной кости, и поворот/сдвиг) размывает тонкий
# сигнал стороны бедра и вызывает путаницу hip_left/hip_right.
# True  -> полная аугментация (как было)
# False -> только CLAHE + resize/pad, без искажений (как в eval)
ENABLE_TRAIN_AUGMENTATION = False  # эксперимент: проверяем гипотезу, что аугментация мешала стороне бедра
# Флип с меной label — отдельный тумблер: это не столько 'искажение',
# сколько способ удвоить обучающие примеры с ЗАВЕДОМО верной меткой
# (зеркальное бедро — валидный пример другой стороны). Обычно стоит
# оставлять True даже при ENABLE_TRAIN_AUGMENTATION=False.
ENABLE_FLIP_AUGMENTATION = True

# Backbone: ImageNet-претрейн через timm (скачивается автоматически при
# первом вызове timm.create_model(..., pretrained=True), кэшируется в
# ~/.cache/torch/hub). Опционально можно заменить на RadImageNet-веса
# (радиологический претрейн, см. https://github.com/BMEII-AI/RadImageNet) —
# для этого нужно вручную скачать .pt/.h5 веса по ссылке из репозитория,
# положить путь в RADIMAGENET_WEIGHTS_PATH и раскомментировать загрузку
# в build_model() ниже. По умолчанию используется ImageNet — самый
# надёжный вариант "из коробки" на Kaggle/Colab без внешних зависимостей.
RADIMAGENET_WEIGHTS_PATH = None  # например "/kaggle/input/radimagenet/resnet18.pt"

LABEL_TO_IDX = {"spine": 0, "hip_left": 1, "hip_right": 2}
IDX_TO_LABEL = {v: k for k, v in LABEL_TO_IDX.items()}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


Device: cuda


## Фиксация окружения (для воспроизводимости и отчётности)

In [23]:
def file_md5(path: str) -> str:
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()


def get_pip_freeze() -> list[str]:
    try:
        out = subprocess.run(["pip", "freeze"], capture_output=True, text=True, check=True)
        return out.stdout.splitlines()
    except Exception as exc:  # noqa: BLE001
        return [f"pip freeze failed: {exc}"]


run_info = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "seed": SEED,
    "python_version": platform.python_version(),
    "platform": platform.platform(),
    "torch_version": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda,
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "timm_version": timm.__version__,
    "albumentations_version": A.__version__,
    "manifest_path": MANIFEST_PATH,
    "manifest_md5": file_md5(MANIFEST_PATH),
    "hyperparameters": {
        "image_size": IMAGE_SIZE,
        "batch_size": BATCH_SIZE,
        "num_epochs": NUM_EPOCHS,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "head_lr": HEAD_LR,
        "backbone_lr": BACKBONE_LR,
        "n_folds": N_FOLDS,
        "backbone": "resnet18 (timm, ImageNet pretrained)" if RADIMAGENET_WEIGHTS_PATH is None else f"resnet18 (RadImageNet: {RADIMAGENET_WEIGHTS_PATH})",
        "enable_train_augmentation": ENABLE_TRAIN_AUGMENTATION,
        "enable_flip_augmentation": ENABLE_FLIP_AUGMENTATION,
    },
    "label_to_idx": LABEL_TO_IDX,
}

with open(os.path.join(OUTPUT_DIR, "run_info.json"), "w", encoding="utf-8") as f:
    json.dump(run_info, f, ensure_ascii=False, indent=2)
with open(os.path.join(OUTPUT_DIR, "pip_freeze.txt"), "w", encoding="utf-8") as f:
    f.write("\n".join(get_pip_freeze()))

print(json.dumps(run_info, ensure_ascii=False, indent=2))


{
  "timestamp_utc": "2026-09-26T17:09:42.904793+00:00",
  "seed": 42,
  "python_version": "3.12.13",
  "platform": "Linux-6.12.90+-x86_64-with-glibc2.35",
  "torch_version": "2.10.0+cu128",
  "cuda_available": true,
  "cuda_version": "12.8",
  "gpu_name": "Tesla T4",
  "timm_version": "1.0.26",
  "albumentations_version": "2.0.8",
  "manifest_path": "/kaggle/input/datasets/cgvhbjmk/dxa-data/training_package/manifest.csv",
  "manifest_md5": "8e4b1368cac77d959b4184c78e73a1ef",
  "hyperparameters": {
    "image_size": 224,
    "batch_size": 16,
    "num_epochs": 40,
    "early_stopping_patience": 7,
    "head_lr": 0.001,
    "backbone_lr": 1e-05,
    "n_folds": 5,
    "backbone": "resnet18 (timm, ImageNet pretrained)",
    "enable_train_augmentation": false,
    "enable_flip_augmentation": true
  },
  "label_to_idx": {
    "spine": 0,
    "hip_left": 1,
    "hip_right": 2
  }
}


## Аугментация

Зеркалит `services/region_classifier/training/augmentation.py` — держим
копию прямо в ноутбуке, чтобы не тащить весь репозиторий в Kaggle/Colab
(ноутбук самодостаточен, нужен только manifest.csv + сами DICOM).

Напоминание по дизайну (подробнее — в докстринге исходного файла):
- горизонтальный флип обрабатывается отдельно и меняет метку `hip_left <-> hip_right`
- НЕ используется вертикальный флип и повороты >12° (угол наклона —
  часть отдельной будущей задачи контроля качества, не должен здесь
  "выучиваться как несущественный")
- контраст/гамма/шум аугментируются заметно сильнее обычного — суррогат
  разброса между разными DXA-аппаратами (Hologic/iDXA), которых в
  обучающих данных физически нет, а организаторы просили generalization


In [24]:
FLIP_LABEL_SWAP = {"hip_left": "hip_right", "hip_right": "hip_left"}


def flip_with_label_swap(image: np.ndarray, label: str, p: float = 0.5):
    if random.random() >= p:
        return image, label
    flipped = cv2.flip(image, 1)
    return flipped, FLIP_LABEL_SWAP.get(label, label)


def clahe(image: np.ndarray) -> np.ndarray:
    return cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8)).apply(image)


def build_train_transform(image_size=IMAGE_SIZE) -> A.Compose:
    return A.Compose([
        A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.8),
        A.RandomGamma(gamma_limit=(70, 130), p=0.5),
        A.OneOf([
            A.GaussNoise(std_range=(0.02, 0.08), p=1.0),
            A.MultiplicativeNoise(multiplier=(0.9, 1.1), p=1.0),
        ], p=0.4),
        A.OneOf([
            A.GaussianBlur(blur_limit=(3, 5), p=1.0),
            A.Sharpen(alpha=(0.1, 0.3), p=1.0),
        ], p=0.3),
        A.Affine(scale=(0.92, 1.08), translate_percent=(0.0, 0.06), rotate=(-12, 12), fit_output=False, p=0.7),
        A.CoarseDropout(num_holes_range=(1, 2), hole_height_range=(0.05, 0.18),
                         hole_width_range=(0.05, 0.18), fill=0, p=0.25),
        A.LongestMaxSize(max_size=image_size),
        A.PadIfNeeded(min_height=image_size, min_width=image_size, border_mode=cv2.BORDER_CONSTANT, fill=0),
    ], seed=SEED)


def build_eval_transform(image_size=IMAGE_SIZE) -> A.Compose:
    return A.Compose([
        A.LongestMaxSize(max_size=image_size),
        A.PadIfNeeded(min_height=image_size, min_width=image_size, border_mode=cv2.BORDER_CONSTANT, fill=0),
    ])


def augment_sample(image_u8: np.ndarray, label: str, train: bool, image_size=IMAGE_SIZE):
    image = clahe(image_u8)
    if train:
        if ENABLE_FLIP_AUGMENTATION:
            image, label = flip_with_label_swap(image, label)
        transform = build_train_transform(image_size) if ENABLE_TRAIN_AUGMENTATION else build_eval_transform(image_size)
    else:
        transform = build_eval_transform(image_size)
    augmented = transform(image=image)["image"]
    return augmented, label


## Dataset и загрузка манифеста

In [25]:
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


class RegionDataset(Dataset):
    # train=True -> аугментация + случайный flip с меной метки.
    # train=False -> детерминированный eval-препроцессинг.

    def __init__(self, df: pd.DataFrame, images_dir: str, train: bool):
        self.df = df.reset_index(drop=True)
        self.images_dir = images_dir
        self.train = train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = os.path.join(self.images_dir, row["image_filename"])
        image_u8 = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if image_u8 is None:
            raise FileNotFoundError(f"Не удалось прочитать {path}")
        image_u8, label = augment_sample(image_u8, row["label"], train=self.train)

        # grayscale -> 3 канала (претрейн ждёт RGB) + ImageNet-нормализация
        rgb = np.stack([image_u8] * 3, axis=-1).astype(np.float32) / 255.0
        rgb = (rgb - IMAGENET_MEAN) / IMAGENET_STD
        tensor = torch.from_numpy(rgb.transpose(2, 0, 1)).float()

        return tensor, LABEL_TO_IDX[label]


manifest = pd.read_csv(MANIFEST_PATH)
print("Всего строк в манифесте:", len(manifest))
print(manifest["label"].value_counts())
print(manifest["fold"].value_counts().sort_index())

# Разделяем на CV-часть (для обучения и подбора модели) и настоящий
# отложенный тест (fold == TEST_FOLD) — эти данные не участвуют ни в
# обучении, ни в выборе лучшей эпохи ни для одного фолда, видим их
# только один раз, в самом конце.
cv_manifest = manifest[manifest["fold"] != TEST_FOLD].reset_index(drop=True)
test_manifest = manifest[manifest["fold"] == TEST_FOLD].reset_index(drop=True)
HELD_OUT_PREFIXES = ("96d715",)   # исключены из обучения и валидации, проверяются только в тесте
_held = cv_manifest["image_filename"].str.startswith(HELD_OUT_PREFIXES)
test_manifest = pd.concat([test_manifest, cv_manifest[_held]]).reset_index(drop=True)
cv_manifest = cv_manifest[~_held].reset_index(drop=True)
print("Вынесено из обучения в тест:", int(_held.sum()), "снимок(ов)")
print(f"\nCV (обучение/валидация): {len(cv_manifest)} изображений")
print(cv_manifest["label"].value_counts())
print(f"\nОтложенный тест (fold {TEST_FOLD}, не используется в обучении): {len(test_manifest)} изображений")
print(test_manifest["label"].value_counts())


Всего строк в манифесте: 252
label
spine        99
hip_left     79
hip_right    74
Name: count, dtype: int64
fold
0    51
1    50
2    51
3    51
4    49
Name: count, dtype: int64
Вынесено из обучения в тест: 1 снимок(ов)

CV (обучение/валидация): 202 изображений
label
spine        80
hip_left     62
hip_right    60
Name: count, dtype: int64

Отложенный тест (fold 4, не используется в обучении): 50 изображений
label
spine        19
hip_left     17
hip_right    14
Name: count, dtype: int64


## Модель: ResNet18 с ImageNet-претрейном, частичная заморозка

In [26]:
def build_model(num_classes: int = 3) -> nn.Module:
    model = timm.create_model("resnet18", pretrained=True, num_classes=num_classes)

    if RADIMAGENET_WEIGHTS_PATH is not None:
        state = torch.load(RADIMAGENET_WEIGHTS_PATH, map_location="cpu", weights_only=False)
        missing, unexpected = model.load_state_dict(state, strict=False)
        print("RadImageNet веса загружены. missing:", missing, "unexpected:", unexpected)

    # Замораживаем ранние слои (общие низкоуровневые фильтры не нужно
    # переучивать на ~250 изображениях) — оставляем обучаемыми layer3, layer4
    # и голову. См. обоснование в docs/plan.md, Шаг 1.
    for name, param in model.named_parameters():
        if name.startswith(("conv1", "bn1", "layer1", "layer2")):
            param.requires_grad = False

    return model


def build_optimizer(model: nn.Module):
    head_params = []
    backbone_params = []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if name.startswith("fc"):
            head_params.append(param)
        else:
            backbone_params.append(param)
    return torch.optim.AdamW([
        {"params": head_params, "lr": HEAD_LR},
        {"params": backbone_params, "lr": BACKBONE_LR},
    ])


## Цикл обучения/валидации на одном фолде

In [27]:
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)

    total_loss = 0.0
    all_preds, all_labels = [], []

    with torch.set_grad_enabled(is_train):
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * images.size(0)
            all_preds.extend(outputs.argmax(dim=1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    f1_macro = f1_score(all_labels, all_preds, average="macro")
    return avg_loss, acc, f1_macro, all_preds, all_labels


def train_one_fold(fold: int, manifest: pd.DataFrame):
    # Пересеиваем перед каждым фолдом — иначе порядок фолдов влиял бы на
    # то, какое RNG-состояние достаётся каждому, и повторный запуск с
    # другим N_FOLDS/порядком дал бы другой результат для того же фолда.
    set_full_determinism(int(SEED + fold))

    train_df = manifest[manifest["fold"] != fold]
    val_df = manifest[manifest["fold"] == fold]

    train_ds = RegionDataset(train_df, IMAGES_DIR, train=True)
    val_ds = RegionDataset(val_df, IMAGES_DIR, train=False)
    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
        drop_last=True, worker_init_fn=worker_init_fn, generator=make_generator(int(SEED + fold)),
    )
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    model = build_model().to(DEVICE)
    optimizer = build_optimizer(model)
    criterion = nn.CrossEntropyLoss()

    best_val_f1 = -1.0
    best_state = None
    epochs_without_improvement = 0
    history = []

    for epoch in range(NUM_EPOCHS):
        train_loss, train_acc, train_f1, _, _ = run_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc, val_f1, val_preds, val_labels = run_epoch(model, val_loader, criterion)

        history.append(dict(
            epoch=epoch + 1, train_loss=train_loss, train_acc=train_acc, train_f1=train_f1,
            val_loss=val_loss, val_acc=val_acc, val_f1=val_f1,
        ))
        print(f"fold {fold} | epoch {epoch+1}/{NUM_EPOCHS} | "
              f"train_loss={train_loss:.4f} train_f1={train_f1:.4f} | "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} val_f1={val_f1:.4f}")

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            best_preds, best_labels = val_preds, val_labels
            best_epoch = epoch + 1
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
                print(f"fold {fold}: early stopping на эпохе {epoch+1}")
                break

    # Чекпоинт сохраняем вместе с конфигом/метаданными — не только веса,
    # чтобы через месяц можно было понять, чем именно это обучено, без
    # необходимости листать историю ноутбука.
    checkpoint = {
        "model_state_dict": best_state,
        "label_to_idx": LABEL_TO_IDX,
        "image_size": IMAGE_SIZE,
        "best_epoch": best_epoch,
        "best_val_f1": best_val_f1,
        "fold": fold,
        "run_info": run_info,
    }
    torch.save(checkpoint, os.path.join(OUTPUT_DIR, f"region_classifier_fold{fold}.pt"))

    pd.DataFrame(history).to_csv(os.path.join(OUTPUT_DIR, f"history_fold{fold}.csv"), index=False)

    return best_val_f1, best_preds, best_labels


def train_final(manifest: pd.DataFrame, num_epochs: int = FINAL_EPOCHS):
    """Одна модель на всех переданных данных, фиксированное число эпох, без валидации и ранней остановки."""
    set_full_determinism(int(SEED))
    ds = RegionDataset(manifest, IMAGES_DIR, train=True)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, drop_last=True,
                        worker_init_fn=worker_init_fn, generator=make_generator(int(SEED)))
    model = build_model().to(DEVICE)
    optimizer = build_optimizer(model)
    criterion = nn.CrossEntropyLoss()
    for epoch in range(num_epochs):
        loss, acc, f1, _, _ = run_epoch(model, loader, criterion, optimizer)
        print(f"epoch {epoch+1}/{num_epochs} | train_loss={loss:.4f} train_acc={acc:.4f} train_f1={f1:.4f}")
    checkpoint = {
        "model_state_dict": {k: v.cpu().clone() for k, v in model.state_dict().items()},
        "label_to_idx": LABEL_TO_IDX, "image_size": IMAGE_SIZE, "epochs": num_epochs, "n_train": len(manifest), "run_info": run_info,
    }
    torch.save(checkpoint, os.path.join(OUTPUT_DIR, "region_classifier.pt"))
    return model


## Запуск 5-fold кросс-валидации

In [28]:
# Одна модель на всех данных (фолды 0-3, без тестового фолда и без вынесенного 96d715); тест — только в конце
print(f"Обучение: {len(cv_manifest)} снимков, {FINAL_EPOCHS} эпох")
final_model = train_final(cv_manifest)


Обучение: 202 снимков, 25 эпох
epoch 1/25 | train_loss=1.0173 train_acc=0.5156 train_f1=0.4128
epoch 2/25 | train_loss=0.8486 train_acc=0.6562 train_f1=0.6111
epoch 3/25 | train_loss=0.7187 train_acc=0.7708 train_f1=0.7279
epoch 4/25 | train_loss=0.5959 train_acc=0.8333 train_f1=0.8068
epoch 5/25 | train_loss=0.5497 train_acc=0.9115 train_f1=0.9007
epoch 6/25 | train_loss=0.4808 train_acc=0.8854 train_f1=0.8643
epoch 7/25 | train_loss=0.4311 train_acc=0.9375 train_f1=0.9281
epoch 8/25 | train_loss=0.4055 train_acc=0.9219 train_f1=0.9096
epoch 9/25 | train_loss=0.3759 train_acc=0.9062 train_f1=0.8913
epoch 10/25 | train_loss=0.3317 train_acc=0.9375 train_f1=0.9302
epoch 11/25 | train_loss=0.3335 train_acc=0.9427 train_f1=0.9373
epoch 12/25 | train_loss=0.2823 train_acc=0.9844 train_f1=0.9824
epoch 13/25 | train_loss=0.2811 train_acc=0.9531 train_f1=0.9487
epoch 14/25 | train_loss=0.2550 train_acc=0.9844 train_f1=0.9828
epoch 15/25 | train_loss=0.1927 train_acc=0.9896 train_f1=0.9879
epo

## Метрики по CV (folds 0..TEST_FOLD-1) — для выбора модели, не финальная оценка

Это честная out-of-fold оценка, но только на CV-части данных (без последнего, отложенного fold'а). Настоящая финальная метрика — на `test_manifest` (последний fold, TEST_FOLD), которая ни разу не участвовала ни в обучении, ни в подборе лучшей эпохи — см. следующий раздел.

In [29]:
target_names = [IDX_TO_LABEL[i] for i in range(len(IDX_TO_LABEL))]
report_text = classification_report(all_val_labels, all_val_preds, target_names=target_names, digits=3)
print(report_text)

cm = confusion_matrix(all_val_labels, all_val_preds)
cm_df = pd.DataFrame(cm, index=target_names, columns=target_names)
print("Confusion matrix (строки — истинные, столбцы — предсказанные):")
print(cm_df)

with open(os.path.join(OUTPUT_DIR, "classification_report.txt"), "w", encoding="utf-8") as f:
    f.write(report_text)
cm_df.to_csv(os.path.join(OUTPUT_DIR, "confusion_matrix.csv"))


              precision    recall  f1-score   support

       spine      1.000     1.000     1.000        80
    hip_left      0.984     0.984     0.984        62
   hip_right      0.983     0.983     0.983        60

    accuracy                          0.990       202
   macro avg      0.989     0.989     0.989       202
weighted avg      0.990     0.990     0.990       202

Confusion matrix (строки — истинные, столбцы — предсказанные):
           spine  hip_left  hip_right
spine         80         0          0
hip_left       0        61          1
hip_right      0         1         59


## Модель для оценки

ОДНА модель: обучена выше на всех данных (фолды 0-3) фиксированное число эпох; `test_manifest` она не видела. Класс называется `EnsembleClassifier`, но в него загружается один чекпоинт `region_classifier.pt`.


In [30]:
class EnsembleClassifier:
    def __init__(self, checkpoint_paths: list[str], device: torch.device = DEVICE):
        self.device = device
        self.models = []
        self.label_to_idx = None
        self.image_size = IMAGE_SIZE

        for path in checkpoint_paths:
            ckpt = torch.load(path, map_location=device, weights_only=False)
            if self.label_to_idx is None:
                self.label_to_idx = ckpt["label_to_idx"]
                self.image_size = ckpt["image_size"]
            elif ckpt["label_to_idx"] != self.label_to_idx:
                raise ValueError(f"{path}: label_to_idx не совпадает с остальными чекпоинтами фолдов")

            model = build_model(num_classes=len(ckpt["label_to_idx"]))
            model.load_state_dict(ckpt["model_state_dict"])
            model.to(device).eval()
            self.models.append(model)

        self.idx_to_label = {v: k for k, v in self.label_to_idx.items()}
        print(f"Загружено моделей в ансамбль: {len(self.models)}")

    def _preprocess(self, image_u8: np.ndarray) -> torch.Tensor:
        # тот же детерминированный eval-препроцессинг, что и на валидации:
        # CLAHE -> resize/pad, БЕЗ случайной аугментации (train=False)
        image, _ = augment_sample(image_u8, label="spine", train=False, image_size=self.image_size)
        rgb = np.stack([image] * 3, axis=-1).astype(np.float32) / 255.0
        rgb = (rgb - IMAGENET_MEAN) / IMAGENET_STD
        tensor = torch.from_numpy(rgb.transpose(2, 0, 1)).float()
        return tensor.unsqueeze(0).to(self.device)

    @torch.no_grad()
    def predict(self, image_u8: np.ndarray) -> dict:
        # image_u8: grayscale uint8 numpy array (как из cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        # или из load_pixel_array для сырого DICOM — препроцессинг тот же).
        tensor = self._preprocess(image_u8)

        per_model_probs = []
        for model in self.models:
            logits = model(tensor)
            probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
            per_model_probs.append(probs)

        per_model_probs = np.stack(per_model_probs, axis=0)  # (n_models, n_classes)
        avg_probs = per_model_probs.mean(axis=0)
        pred_idx = int(avg_probs.argmax())

        return {
            "label": self.idx_to_label[pred_idx],
            "confidence": float(avg_probs[pred_idx]),
            "probs_by_class": {self.idx_to_label[i]: float(p) for i, p in enumerate(avg_probs)},
            "per_model_agreement": float((per_model_probs.argmax(axis=1) == pred_idx).mean()),
        }


# Пример использования на новом снимке (замените путь на реальный):
#
# ensemble = EnsembleClassifier([
#     os.path.join(OUTPUT_DIR, f"region_classifier_fold{i}.pt") for i in range(N_FOLDS)
# ])
# new_image = cv2.imread("/path/to/new_image.png", cv2.IMREAD_GRAYSCALE)
# result = ensemble.predict(new_image)
# print(result)
#
# per_model_agreement < 1.0 — сигнал разногласия моделей между собой,
# полезно как дополнительная мера неуверенности (аналогично
# region_confidence/side_confidence в classify.py) — если модели
# ансамбля расходятся, стоит пометить снимок на ручную проверку.

## Финальная оценка на настоящем отложенном тесте (внутри наших данных)

Это главная метрика. `test_manifest` (fold == TEST_FOLD и вынесенный снимок 96d715) не участвовала в обучении модели.


In [31]:
ensemble = EnsembleClassifier([os.path.join(OUTPUT_DIR, "region_classifier.pt")])   # одна модель

test_preds, test_labels_true, test_confidences, test_agreements = [], [], [], []
for _, row in test_manifest.iterrows():
    img = cv2.imread(os.path.join(IMAGES_DIR, row["image_filename"]), cv2.IMREAD_GRAYSCALE)
    result = ensemble.predict(img)
    test_preds.append(LABEL_TO_IDX[result["label"]])
    test_labels_true.append(LABEL_TO_IDX[row["label"]])
    test_confidences.append(result["confidence"])
    test_agreements.append(result["per_model_agreement"])

test_report = classification_report(test_labels_true, test_preds, target_names=target_names, digits=3)
print(f"Отложенный тест: {len(test_manifest)} изображений (никогда не участвовали в обучении)")
print(test_report)

test_cm = confusion_matrix(test_labels_true, test_preds)
test_cm_df = pd.DataFrame(test_cm, index=target_names, columns=target_names)
print("Confusion matrix (отложенный тест):")
print(test_cm_df)
print(f"\nСредняя уверенность ансамбля: {np.mean(test_confidences):.3f}, "
      f"среднее согласие моделей между собой: {np.mean(test_agreements):.3f}")

with open(os.path.join(OUTPUT_DIR, "holdout_test_classification_report.txt"), "w", encoding="utf-8") as f:
    f.write(test_report)
test_cm_df.to_csv(os.path.join(OUTPUT_DIR, "holdout_test_confusion_matrix.csv"))


Загружено моделей в ансамбль: 1
Отложенный тест: 50 изображений (никогда не участвовали в обучении)
              precision    recall  f1-score   support

       spine      1.000     1.000     1.000        19
    hip_left      1.000     0.941     0.970        17
   hip_right      0.933     1.000     0.966        14

    accuracy                          0.980        50
   macro avg      0.978     0.980     0.978        50
weighted avg      0.981     0.980     0.980        50

Confusion matrix (отложенный тест):
           spine  hip_left  hip_right
spine         19         0          0
hip_left       0        16          1
hip_right      0         0         14

Средняя уверенность ансамбля: 0.924, среднее согласие моделей между собой: 1.000


## Голосование критериев: сеть + математика (маска кости) + ширина снимка

Регион (позвоночник / бедро) — по большинству голосов; ничья решается сетью. Сторона бедра — у сети, если её регион совпал с итоговым, иначе по смещению массы кости. Ниже: точность каждого критерия, итога и все снимки, где критерии разошлись.

In [32]:
"""Голосование критериев для классификатора региона: сеть + математика (маска кости и периодичность, classify.py) + ширина снимка.
Самодостаточный файл (numpy, cv2, scipy): этот же текст встраивается в ноутбук train_region_classifier_vote.ipynb (build_train_region_vote_nb.py), одна версия кода на всё.

Голоса по региону (spine | hip):
  1) сеть: класс из {spine, hip_left, hip_right} -> регион; уверенность сети отдельно;
  2) математика: маска кости (CLAHE + Otsu + наибольшая компонента) -> число вертикальных пиков ширины кости: >= 6 позвоночник, иначе бедро; сторона бедра по смещению массы кости в верхней трети кадра;
  3) ширина снимка: 300±5 -> spine, 280±5 -> hip, другая ширина -> воздерживается (2 из 252 бёдер имеют 248).
Итог: регион, за который проголосовало не меньше двух; при ничьей (один воздержался) решает сеть, а без сети — математика. Сторона бедра: если сеть и математика согласны — она; при разногласии побеждает тот, у кого выше уверенность (у сети — её вероятность, у математики 0.5 + 0.7·|смещение массы|); если регион сети не совпал с итоговым — математика.
"""
import cv2
import numpy as np
from scipy.signal import find_peaks

SPINE_WIDTHS = (300,)
HIP_WIDTHS = (280,)
WIDTH_TOL = 5                   # допуск по ширине, px
CLAHE_CLIP_LIMIT = 3.0
CLAHE_TILE_GRID = (8, 8)
PEAK_SMOOTH_WINDOW = 7
PEAK_PROMINENCE = 0.05
PEAK_MIN_DISTANCE = 6
SPINE_PEAK_THRESHOLD = 6        # >= этого числа пиков -> позвоночник
SIDE_TOP_FRACTION = 0.35        # верхняя часть кадра, по которой считается смещение массы кости
SIDE_CONF_SCALE = 0.7           # уверенность стороны по математике = 0.5 + 0.7·|смещение| (как side_margin_confidence в classify.py)


def segment_bone(image_u8):
    """Бинарная маска (0/255) наибольшей связной светлой области."""
    eq = cv2.createCLAHE(clipLimit=CLAHE_CLIP_LIMIT, tileGridSize=CLAHE_TILE_GRID).apply(image_u8)
    _, th = cv2.threshold(cv2.GaussianBlur(eq, (5, 5), 0), 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    k = np.ones((3, 3), np.uint8)
    th = cv2.morphologyEx(cv2.morphologyEx(th, cv2.MORPH_OPEN, k), cv2.MORPH_CLOSE, k)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(th, connectivity=8)
    if n <= 1:
        return th
    return (labels == 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])).astype("uint8") * 255


def count_vertical_peaks(mask):
    w = (mask > 0).sum(axis=1).astype(float)
    if w.max() <= 0:
        return 0
    sm = np.convolve(w, np.ones(PEAK_SMOOTH_WINDOW) / PEAK_SMOOTH_WINDOW, mode="same")
    return len(find_peaks(sm / sm.max(), prominence=PEAK_PROMINENCE, distance=PEAK_MIN_DISTANCE)[0])


def mask_vote(img):
    """Математика: регион по периодичности кости и сторона бедра (image_left | image_right) по смещению массы в верхней части."""
    mask = segment_bone(img)
    n_peaks = count_vertical_peaks(mask)
    region = "spine" if n_peaks >= SPINE_PEAK_THRESHOLD else "hip"
    ys, xs = np.nonzero(mask[: int(mask.shape[0] * SIDE_TOP_FRACTION)])
    score = None if len(xs) == 0 else float((xs.mean() - mask.shape[1] / 2) / (mask.shape[1] / 2))
    side = None if score is None else ("image_left" if score < 0 else "image_right")
    return dict(region=region, n_peaks=int(n_peaks), side=side, side_score=score)


def width_vote(img):
    w = img.shape[1]
    if any(abs(w - x) <= WIDTH_TOL for x in SPINE_WIDTHS):
        return "spine"
    return "hip" if any(abs(w - x) <= WIDTH_TOL for x in HIP_WIDTHS) else None


def _region_of(label):
    return None if label is None else ("spine" if label == "spine" else "hip")


def vote(img, cnn_label=None, cnn_conf=None):
    """cnn_label: spine | hip_left | hip_right (или None, если модели нет). Возвращает итоговый класс, регион, стороны, голоса и согласие."""
    m = mask_vote(img)
    votes = {"cnn": _region_of(cnn_label), "mask": m["region"], "width": width_vote(img)}
    active = {k: v for k, v in votes.items() if v is not None}
    counts = {r: sum(1 for v in active.values() if v == r) for r in ("spine", "hip")}
    if counts["spine"] != counts["hip"]:
        region = max(counts, key=counts.get)
    else:                                                    # ничья: сеть, а без сети — математика
        region = votes["cnn"] or votes["mask"]
    agreement = counts[region] / len(active)
    cnn_side = {"hip_left": "image_left", "hip_right": "image_right"}.get(cnn_label)
    side = None
    if region == "hip":
        mask_conf = None if m["side_score"] is None else min(0.99, 0.5 + SIDE_CONF_SCALE * abs(m["side_score"]))     # уверенность математики: чем дальше масса кости от центра, тем увереннее
        if cnn_side and m["side"] and cnn_side != m["side"] and cnn_conf is not None:
            side = cnn_side if float(cnn_conf) >= mask_conf else m["side"]        # сторона: при разногласии побеждает тот, кто увереннее
        else:
            side = cnn_side if (votes["cnn"] == "hip" and cnn_side) else m["side"]
    label = "spine" if region == "spine" else ("hip_left" if side == "image_left" else "hip_right")
    conf = agreement if cnn_conf is None else 0.5 * agreement + 0.5 * float(cnn_conf)
    return dict(label=label, region=region, side=side, votes=votes, n_peaks=m["n_peaks"], side_mask=m["side"], agreement=float(agreement),
                unanimous=len(set(active.values())) == 1, confidence=float(conf))

In [33]:
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

rows = []
for _, row in test_manifest.iterrows():
    img = cv2.imread(os.path.join(IMAGES_DIR, row["image_filename"]), cv2.IMREAD_GRAYSCALE)
    r_ = ensemble.predict(img)
    v_ = vote(img, r_["label"], r_["confidence"])
    rows.append(dict(image_filename=row["image_filename"], true=row["label"], cnn=r_["label"], cnn_conf=r_["confidence"], vote_label=v_["label"],
                     v_cnn=v_["votes"]["cnn"], v_mask=v_["votes"]["mask"], v_width=v_["votes"]["width"], n_peaks=v_["n_peaks"], agreement=v_["agreement"]))
vd = pd.DataFrame(rows)
vd["true_region"] = vd.true.map(lambda x: "spine" if x == "spine" else "hip")
labels_ = ["spine", "hip_left", "hip_right"]
print("Точность по РЕГИОНУ (позвоночник / бедро) и по классу из трёх:")
for name, col_r, col_c in (("сеть", "v_cnn", "cnn"), ("математика (маска кости)", "v_mask", None), ("ширина снимка", "v_width", None), ("ГОЛОСОВАНИЕ", None, "vote_label")):
    if col_r:
        used = vd[col_r].notna()
        reg = f"{(vd.loc[used, col_r] == vd.loc[used, 'true_region']).mean():.3f} ({int(used.sum())}/{len(vd)} снимков с голосом)"
    else:
        reg = f"{(vd.vote_label.map(lambda x: 'spine' if x == 'spine' else 'hip') == vd.true_region).mean():.3f}"
    cls = f"{(vd[col_c] == vd.true).mean():.3f}" if col_c else "—"
    print(f"  {name:28s} регион: {reg} | класс из 3: {cls}")
print("\n=== ИТОГ ГОЛОСОВАНИЯ ===")
print(classification_report(vd.true, vd.vote_label, labels=labels_, digits=3, zero_division=0))
print(pd.DataFrame(confusion_matrix(vd.true, vd.vote_label, labels=labels_), index=labels_, columns=labels_))
bad = vd[(vd.cnn != vd.true) | (vd.vote_label != vd.true) | (vd.agreement < 1)]
print(f"\nСнимки, где критерии разошлись или ответ неверный: {len(bad)}")
print(bad[["image_filename", "true", "cnn", "cnn_conf", "v_mask", "v_width", "n_peaks", "vote_label"]].round(3).to_string(index=False))
held_ = vd[vd.image_filename.str.startswith(HELD_OUT_PREFIXES)]
print("Вынесенный из обучения снимок (проверка только на нём):")
print(held_[["image_filename", "true", "cnn", "cnn_conf", "v_mask", "v_width", "n_peaks", "vote_label"]].round(3).to_string(index=False))
vd.to_csv(os.path.join(OUTPUT_DIR, "region_votes_test.csv"), index=False)

Точность по РЕГИОНУ (позвоночник / бедро) и по классу из трёх:
  сеть                         регион: 1.000 (50/50 снимков с голосом) | класс из 3: 0.980
  математика (маска кости)     регион: 1.000 (50/50 снимков с голосом) | класс из 3: —
  ширина снимка                регион: 1.000 (50/50 снимков с голосом) | класс из 3: —
  ГОЛОСОВАНИЕ                  регион: 1.000 | класс из 3: 1.000

=== ИТОГ ГОЛОСОВАНИЯ ===
              precision    recall  f1-score   support

       spine      1.000     1.000     1.000        19
    hip_left      1.000     1.000     1.000        17
   hip_right      1.000     1.000     1.000        14

    accuracy                          1.000        50
   macro avg      1.000     1.000     1.000        50
weighted avg      1.000     1.000     1.000        50

           spine  hip_left  hip_right
spine         19         0          0
hip_left       0        17          0
hip_right      0         0         14

Снимки, где критерии разошлись или ответ неверн

## Дополнительная проверка: внешний скан-сеанс ("Для теста", n=3)

Основная финальная метрика — уже выше, на 50 отложенных изображениях из нашего датасета. Этот раздел — дополнительный, не главный, sanity-check: 3 файла из отдельной папки организаторов "Для теста", снятые в другой скан-сессии (не входят в наш датасет вообще). Метки здесь — из имени файла (`_ПОП`/`_ППОБ`/`_ЛПОБ`), не из нашей ручной проверки. Полезно как ещё один независимый источник, но n=3 всё равно мало для статистики — смотрим на это как на дополнительное подтверждение, а не отдельную метрику.

In [34]:
# --- ПРАВЬТЕ ПОД СВОЮ СРЕДУ ---
HOLDOUT_IMAGES_DIR = "/kaggle/input/holdout-test-package/images"
HOLDOUT_MANIFEST_PATH = "/kaggle/input/holdout-test-package/manifest_holdout.csv"

if os.path.exists(HOLDOUT_MANIFEST_PATH):
    ensemble = EnsembleClassifier([
        os.path.join(OUTPUT_DIR, "region_classifier.pt")
    ])

    holdout = pd.read_csv(HOLDOUT_MANIFEST_PATH)
    correct = 0
    for _, row in holdout.iterrows():
        img = cv2.imread(os.path.join(HOLDOUT_IMAGES_DIR, row["image_filename"]), cv2.IMREAD_GRAYSCALE)
        result = ensemble.predict(img)
        is_correct = result["label"] == row["true_label"]
        correct += is_correct
        print(f"{row['source_filename']}: true={row['true_label']} pred={result['label']} "
              f"(conf={result['confidence']:.3f}, agreement={result['per_model_agreement']:.2f}) "
              f"{'OK' if is_correct else 'MISMATCH'}")

    print(f"\n{correct}/{len(holdout)} совпадений с истинной разметкой (n слишком мал для статистики)")
else:
    print("Внешнего пакета нет — необязательную проверку пропускаем")


Внешнего пакета нет — необязательную проверку пропускаем


## Что дальше

- В `./outputs/` сохранено всё для воспроизводимости и аудита:
  `region_classifier_fold{N}.pt` (веса + конфиг + label_to_idx + run_info
  внутри одного файла), `history_fold{N}.csv` (метрики по эпохам),
  `run_info.json` (версии библиотек, seed, hash манифеста, гиперпараметры),
  `pip_freeze.txt` (полный список версий пакетов), `cv_summary.json`,
  `classification_report.txt`, `confusion_matrix.csv`.
- Для инференса можно усреднить предсказания всех 5 фолдов (ensemble) или
  взять фолд с лучшим val-F1 (см. `best_val_f1` внутри чекпоинта).
- Если результат не превосходит OpenCV-фолбэк (99.6% на прокси-оценке,
  см. `services/region_classifier/README.md`) — это ожидаемо на таком
  маленьком датасете; ценность этой модели — в потенциальной
  генерализации на снимки других DXA-аппаратов за счёт агрессивной
  контрастной аугментации, которую нельзя оценить без реальных
  multi-vendor данных (см. обсуждение датасетов UK Biobank / VerteNet /
  Bone Densitometry Dataset в истории работы над проектом).
- Следующий шаг по плану (`docs/plan.md`) — Шаг 2: keypoint-модель для
  критериев укладки/оси/ROI.
